In [19]:
import os
import pandas as pd
import pyodbc



In [20]:
SERVER   = "DEVICE-HP"      
DATABASE = "SalesDWH"       

def connect():
    return pyodbc.connect(
        f"DRIVER={{ODBC Driver 17 for SQL Server}};"
        f"SERVER={SERVER};"
        f"DATABASE={DATABASE};"
        f"Trusted_Connection=yes;"
    )


CSV_FOLDER = r"E:\D\Document\DA & BI Hub\Database\DataWarehouse_Project\dataset" 
BULK_FILES = ["Sales 2022.csv", "Sales 2023.csv", "Sales 2024.csv", "Sales 2025.csv"]


In [21]:
def create_tables(conn):
    conn.execute("""
        IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='All_Raw_sales' AND xtype='U')
        CREATE TABLE All_Raw_sales (
            order_id      NVARCHAR(30),
            order_date    NVARCHAR(20),
            ship_date     NVARCHAR(20),
            ship_mode     NVARCHAR(50),
            customer_id   NVARCHAR(20),
            customer_name NVARCHAR(100),
            segment       NVARCHAR(50),
            country       NVARCHAR(100),
            city          NVARCHAR(100),
            state         NVARCHAR(100),
            postal_code   NVARCHAR(20),
            region        NVARCHAR(50),
            product_id    NVARCHAR(30),
            category      NVARCHAR(100),
            sub_category  NVARCHAR(100),
            product_name  NVARCHAR(255),
            sales         DECIMAL(12,4),
            quantity      INT,
            discount      DECIMAL(5,4),
            profit        DECIMAL(12,4),
            source_file   NVARCHAR(100)
             ) 
        """)

    conn.execute( """
        IF NOT EXISTS (SELECT * FROM sysobjects WHERE name='loaded_files' AND xtype='U')
        CREATE TABLE loaded_files (
            file_name   NVARCHAR(100) PRIMARY KEY,
            load_type  NVARCHAR(20),
            loaded_at   DATETIME DEFAULT GETDATE()
        )
            """)
    conn.commit()
    print("✅ Tables are Ready!"   )


In [22]:
def already_loaded(conn, file_name):
    result = conn.execute(
        "SELECT 1 FROM loaded_files WHERE file_name = ?", (file_name,)
    ).fetchone()
    return result is not None



In [23]:
def read_csv(file_path, file_name):
    df = pd.read_csv(file_path, encoding="utf-8-sig", dtype=str)
    df.columns = [c.strip() for c in df.columns]
    df["source_file"] = file_name
    df = df.fillna("") 
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()
    return df

def to_rows(df):
    return [
        (
            row.get("Order ID",""),      row.get("Order Date",""),
            row.get("Ship Date",""),     row.get("Ship Mode",""),
            row.get("Customer ID",""),   row.get("Customer Name",""),
            row.get("Segment",""),       row.get("Country",""),
            row.get("City",""),          row.get("State",""),
            row.get("Postal Code",""),   row.get("Region",""),
            row.get("Product ID",""),    row.get("Category",""),
            row.get("Sub-Category",""),  row.get("Product Name",""),
            row.get("Sales",""),         row.get("Quantity",""),
            row.get("Discount",""),      row.get("Profit",""),
            str(row.get("source_file",""))
        )
        for _, row in df.iterrows()
    ]
     
INSERT_SQL = """
    INSERT INTO All_Raw_sales VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)
"""    

In [24]:
def bulk_load(conn, file_path):
    file_name = os.path.basename(file_path)

    if already_loaded(conn, file_name):
        print(f"⏭️  {file_name} — already loaded, skipping")
        return

    print(f"📦 Bulk loading: {file_name} ...")
    df   = read_csv(file_path, file_name)
    rows = to_rows(df)

    cursor = conn.cursor()
    # بنقسم على batches عشان نتجنب memory issues
    batch_size = 1000
    for i in range(0, len(rows), batch_size):
        cursor.executemany(INSERT_SQL, rows[i:i+batch_size])
        print(f"  → {min(i+batch_size, len(rows))}/{len(rows)} rows...")

    conn.execute(
        "INSERT INTO loaded_files (file_name, load_type) VALUES (?,?)",
        (file_name, "BULK")
    )
    conn.commit()
    print(f"✅ {file_name} — bulk done! ({len(df)} rows)\n")




In [27]:
# ── Incremental — لأي ملف جديد بعدين ──
def incremental_load(conn, file_path):
    file_name = os.path.basename(file_path)
    
    print(f"🔄 Incremental loading: {file_name} ...")

    # آخر order_date محمّلة في الجدول
    result = conn.execute(
        "SELECT MAX(TRY_CONVERT(DATE, order_date, 23)) FROM All_Raw_sales"
    ).fetchone()
    last_date = result[0]
    print(f"  → Last loaded date: {last_date}")

    # اقرأ الـ CSV
    df = pd.read_csv(file_path, encoding="utf-8-sig", dtype=str)
    df.columns = [c.strip() for c in df.columns]
    df["source_file"] = file_name
    df = df.fillna("")
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip()

    # حوّل order_date لـ datetime عشان نقدر نقارن
    df["Order Date"] = pd.to_datetime(df["Order Date"], infer_datetime_format=True, errors="coerce")

    # خد بس الـ rows الجديدة
    if last_date:
        df = df[df["Order Date"].dt.date > last_date]

    if df.empty:
        print(f"  ℹ️ No new rows to load")
        return

    print(f"  → Found {len(df)} new rows")

    # رجّع order_date لـ string تاني عشان الـ INSERT
    df["Order Date"] = df["Order Date"].dt.strftime("%m/%d/%Y")

    rows = to_rows(df)
    cursor = conn.cursor()
    cursor.executemany(INSERT_SQL, rows)

    conn.commit()
    print(f"✅ {file_name} — incremental done! ({len(df)} new rows)\n")


In [29]:
def main():
    conn = connect()
    print(f"✅ Connected to {SERVER} → {DATABASE}")

    create_tables(conn)

    files = sorted([f for f in os.listdir(CSV_FOLDER) if f.endswith(".csv")])
    print(f"\n📁 Found {len(files)} CSV file(s)\n")

    for f in files:
        file_path= os.path.join(CSV_FOLDER, f)
        if f in BULK_FILES:
            bulk_load(conn, file_path)     
        else:
            incremental_load(conn, file_path) 

    print("\n🎉 Done! All files loaded into All_Raw_sales")
    conn.close()
main()



✅ Connected to DEVICE-HP → SalesDWH
✅ Tables are Ready!

📁 Found 4 CSV file(s)

📦 Bulk loading: Sales 2022.csv ...
  → 1000/1993 rows...
  → 1993/1993 rows...
✅ Sales 2022.csv — bulk done! (1993 rows)

📦 Bulk loading: Sales 2023.csv ...
  → 1000/2102 rows...
  → 2000/2102 rows...
  → 2102/2102 rows...
✅ Sales 2023.csv — bulk done! (2102 rows)

📦 Bulk loading: Sales 2024.csv ...
  → 1000/2587 rows...
  → 2000/2587 rows...
  → 2587/2587 rows...
✅ Sales 2024.csv — bulk done! (2587 rows)

📦 Bulk loading: Sales 2025.csv ...
  → 1000/3312 rows...
  → 2000/3312 rows...
  → 3000/3312 rows...
  → 3312/3312 rows...
✅ Sales 2025.csv — bulk done! (3312 rows)


🎉 Done! All files loaded into All_Raw_sales
